In [ ]:
import json
import time
from datetime import datetime

import pandas as pd
import requests

MAPPING_PATH = "b1a_mapping.csv"
BLS_API_KEY = "12de604065914ed48cc3b31f0fc15d88"

START_YEAR = 2000
END_YEAR = datetime.now().year

BLS_API_V2_URL = "https://api.bls.gov/publicAPI/v2/timeseries/data/"


def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]


def fetch_bls_monthly_windowed(
    series_ids,
    start_year,
    end_year,
    api_key,
    series_batch_size=50,
    year_window=10,
    sleep_s=0.25
):
    """
    Pull monthly data by (series batches) x (year windows) to avoid the ~240 obs cap.
    Returns tidy df: series_id, date, value
    """
    headers = {"Content-type": "application/json"}
    out = []

    for y0 in range(start_year, end_year + 1, year_window):
        y1 = min(end_year, y0 + year_window - 1)

        for batch in chunks(list(series_ids), series_batch_size):
            payload = {
                "seriesid": batch,
                "startyear": str(y0),
                "endyear": str(y1),
                "registrationkey": api_key,
            }

            resp = requests.post(BLS_API_V2_URL, data=json.dumps(payload), headers=headers, timeout=60)
            resp.raise_for_status()
            j = resp.json()

            if j.get("status") != "REQUEST_SUCCEEDED":
                raise RuntimeError(f"BLS API error: status={j.get('status')} message={j.get('message')}")

            for s in j["Results"]["series"]:
                sid = s["seriesID"]
                for item in s.get("data", []):
                    period = item.get("period", "")
                    if period.startswith("M") and period != "M13":
                        year = int(item["year"])
                        month = int(period[1:])
                        date = pd.Timestamp(year=year, month=month, day=1)
                        out.append({"series_id": sid, "date": date, "value": float(item["value"])})

            time.sleep(sleep_s)

    df = pd.DataFrame(out)
    if df.empty:
        return df

    # De-duplicate in case windows overlap or API repeats endpoints
    df = df.drop_duplicates(subset=["series_id", "date"]).sort_values(["series_id", "date"]).reset_index(drop=True)
    return df


# ---- main ----
mapping = pd.read_csv(MAPPING_PATH, dtype=str)

if "series_id" not in mapping.columns:
    raise KeyError(f"'series_id' not found in {MAPPING_PATH}. Columns are: {list(mapping.columns)}")

series_ids = mapping["series_id"].dropna().astype(str).str.strip().unique().tolist()

tidy = fetch_bls_monthly_windowed(
    series_ids=series_ids,
    start_year=START_YEAR,
    end_year=END_YEAR,
    api_key=BLS_API_KEY,
    series_batch_size=50,
    year_window=10,   
    sleep_s=0.25
)

# enforce >= 2000-01
tidy = tidy[tidy["date"] >= pd.Timestamp(START_YEAR, 1, 1)].copy()

wide = (
    tidy.pivot_table(index="date", columns="series_id", values="value", aggfunc="first")
        .sort_index()
)

print("Series count:", len(series_ids))
print("Date range:", wide.index.min(), "to", wide.index.max())
print("Wide shape:", wide.shape)

wide.to_csv("b1a_wide_seriesid.csv")
try:
    wide.to_parquet("b1a_wide_seriesid.parquet")
except ImportError:
    print("Note: pyarrow not installed, skipping parquet output.")